Este proyecto propone construir un modelo de clasificación supervisada que, a partir de datos demográficos y socioeconómicos de una persona adulta (edad, nivel educativo, ocupación, estado civil, país de origen, etc.), prediga si dicha persona ganará más o menos de 50,000 USD al año.

En base a los resultados del modelo, los estudiantes deberán desarrollar un sistema de recomendación interpretativo, capaz de sugerir posibles estrategias o cambios para aumentar la probabilidad de superar ese umbral de ingresos.

Objetivos

Explorar los datos del censo.

Construir perfiles socioeconómicos.

Explorar la importancia y peso de variables sociales (educación, género, raza, etc.) en predicciones económicas.

Aplicar técnicas de sistemas de recomendación.

Visualizar y comunicar hallazgos de forma profesional.

📝 Instrucciones

Carga del conjunto de datos. Usaremos el dataset Adult Income Dataset, también conocido como "Census Income" este información fue recolectada por la Oficina del Censo de EE.UU. y descargada por la academia para guardarla en esta carpeta de proyecto bajo el nombre adult-census-income.csv o puedes cargarlo en el código directamente desde el siguente enlace:# Explore here

1. EDA

In [201]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.neighbors import NearestNeighbors
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import OneHotEncoder
import warnings
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report


url="https://breathecode.herokuapp.com/asset/internal-link?id=2326&path=adult-census-income.csv"

df=pd.read_csv(url)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 32561 entries, 0 to 32560
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   age             32561 non-null  int64
 1   workclass       32561 non-null  str  
 2   fnlwgt          32561 non-null  int64
 3   education       32561 non-null  str  
 4   education.num   32561 non-null  int64
 5   marital.status  32561 non-null  str  
 6   occupation      32561 non-null  str  
 7   relationship    32561 non-null  str  
 8   race            32561 non-null  str  
 9   sex             32561 non-null  str  
 10  capital.gain    32561 non-null  int64
 11  capital.loss    32561 non-null  int64
 12  hours.per.week  32561 non-null  int64
 13  native.country  32561 non-null  str  
 14  income          32561 non-null  str  
dtypes: int64(6), str(9)
memory usage: 3.7 MB


In [202]:
df.info()
print("="*80)
df.isnull().sum()

<class 'pandas.DataFrame'>
RangeIndex: 32561 entries, 0 to 32560
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   age             32561 non-null  int64
 1   workclass       32561 non-null  str  
 2   fnlwgt          32561 non-null  int64
 3   education       32561 non-null  str  
 4   education.num   32561 non-null  int64
 5   marital.status  32561 non-null  str  
 6   occupation      32561 non-null  str  
 7   relationship    32561 non-null  str  
 8   race            32561 non-null  str  
 9   sex             32561 non-null  str  
 10  capital.gain    32561 non-null  int64
 11  capital.loss    32561 non-null  int64
 12  hours.per.week  32561 non-null  int64
 13  native.country  32561 non-null  str  
 14  income          32561 non-null  str  
dtypes: int64(6), str(9)
memory usage: 3.7 MB


age               0
workclass         0
fnlwgt            0
education         0
education.num     0
marital.status    0
occupation        0
relationship      0
race              0
sex               0
capital.gain      0
capital.loss      0
hours.per.week    0
native.country    0
income            0
dtype: int64

In [203]:
df.shape

(32561, 15)

In [204]:
df.describe()

,age,fnlwgt,education.num,capital.gain,capital.loss,hours.per.week
count,32561.000000,3.256100e+04,32561.000000,32561.000000,32561.000000,32561.000000
mean,38.581647,1.897784e+05,10.080679,1077.648844,87.303830,40.437456
std,13.640433,1.055500e+05,2.572720,7385.292085,402.960219,12.347429
min,17.000000,1.228500e+04,1.000000,0.000000,0.000000,1.000000
25%,28.000000,1.178270e+05,9.000000,0.000000,0.000000,40.000000
50%,37.000000,1.783560e+05,10.000000,0.000000,0.000000,40.000000
75%,48.000000,2.370510e+05,12.000000,0.000000,0.000000,45.000000
max,90.000000,1.484705e+06,16.000000,99999.000000,4356.000000,99.000000


In [205]:
duplicados = df.duplicated()
num_duplicados = duplicados.sum()

print(f"El dataset tiene {num_duplicados} filas duplicadas")

df_duplicados = df[duplicados]
print("Filas duplicadas:")
print(df_duplicados)

# Eliminar duplicados
#usar el método drop.duplicated() para eliminar filas duplicadas

df = df.drop_duplicates().reset_index(drop=True)

print ("="*80)

print(f"Shape sin duplicados: {df.shape}")

El dataset tiene 24 filas duplicadas
Filas duplicadas:
       age         workclass  fnlwgt     education  education.num  \
8453    25           Private  308144     Bachelors             13   
8645    90           Private   52386  Some-college             10   
12202   21           Private  250051  Some-college             10   
14346   20           Private  107658  Some-college             10   
15603   25           Private  195994       1st-4th              2   
17344   21           Private  243368     Preschool              1   
19067   46           Private  173243       HS-grad              9   
20388   30           Private  144593       HS-grad              9   
20507   19           Private   97261       HS-grad              9   
22783   19           Private  138153  Some-college             10   
22934   19           Private  146679  Some-college             10   
23276   49           Private   31267       7th-8th              4   
23660   25           Private  195994       1st-4

In [206]:
df["workclass"].unique()

<StringArray>
[               '?',          'Private',        'State-gov',
      'Federal-gov', 'Self-emp-not-inc',     'Self-emp-inc',
        'Local-gov',      'Without-pay',     'Never-worked']
Length: 9, dtype: str

Se visualizan valores "?", se puede reemplazar con nulos

In [207]:
df = df.replace("?", np.nan)

In [208]:
df.isnull().sum()

age                  0
workclass         1836
fnlwgt               0
education            0
education.num        0
marital.status       0
occupation        1843
relationship         0
race                 0
sex                  0
capital.gain         0
capital.loss         0
hours.per.week       0
native.country     582
income               0
dtype: int64

Se elimina las filas que tienen null ya que equivalen el 7% del dataset.

In [209]:
df = df.dropna()

In [210]:
df.shape

(30139, 15)

In [211]:
df.info()

<class 'pandas.DataFrame'>
Index: 30139 entries, 1 to 32536
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   age             30139 non-null  int64
 1   workclass       30139 non-null  str  
 2   fnlwgt          30139 non-null  int64
 3   education       30139 non-null  str  
 4   education.num   30139 non-null  int64
 5   marital.status  30139 non-null  str  
 6   occupation      30139 non-null  str  
 7   relationship    30139 non-null  str  
 8   race            30139 non-null  str  
 9   sex             30139 non-null  str  
 10  capital.gain    30139 non-null  int64
 11  capital.loss    30139 non-null  int64
 12  hours.per.week  30139 non-null  int64
 13  native.country  30139 non-null  str  
 14  income          30139 non-null  str  
dtypes: int64(6), str(9)
memory usage: 3.7 MB


In [212]:
df.income.value_counts()

income
<=50K    22633
>50K      7506
Name: count, dtype: int64

Pasar la variable objetivo a variable binaria

In [213]:
df["income"] = df["income"].map({
    "<=50K": 0,
    ">50K": 1
})

Conclusiones:

La variable objetivo ssería income, se visualiza que hay más personas que tienen ingresos <=50K.
El dataset tiene 8 variables categóricas y 6 variables numéricas.

Procesamiento de datos

In [214]:
X=df.drop("income",axis=1)   #variable predictora
y=df["income"]               #variable objetivo


X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

X_train.head()

,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country
11356,22,Private,188950,Some-college,10,Never-married,Tech-support,Own-child,White,Male,0,0,40,United-States
11749,49,Self-emp-not-inc,155489,Some-college,10,Married-civ-spouse,Craft-repair,Husband,White,Male,0,0,65,Poland
10504,27,Private,220754,Doctorate,16,Never-married,Adm-clerical,Own-child,White,Female,0,0,40,United-States
7927,52,Federal-gov,129177,HS-grad,9,Divorced,Adm-clerical,Not-in-family,White,Female,0,0,40,United-States
18297,57,Self-emp-inc,244605,Some-college,10,Married-civ-spouse,Sales,Husband,White,Male,0,0,50,United-States


In [215]:
X_train.shape, X_test.shape

((24111, 14), (6028, 14))

Escalado

In [216]:
num_variables = [
    "age",
    "fnlwgt",
    "education.num",
    "capital.gain",
    "capital.loss",
    "hours.per.week"
]

In [217]:
# instancio el escalador
scaler = StandardScaler()

# entreno el escalador con los datos de entrenamiento
scaler.fit(X_train[num_variables])

# aplico el escalador en amhos
X_train_num_scal = scaler.transform(X_train[num_variables])
X_train_num_scal = pd.DataFrame(X_train_num_scal, index = X_train.index, columns = num_variables)

X_test_num_scal = scaler.transform(X_test[num_variables])
X_test_num_scal = pd.DataFrame(X_test_num_scal, index = X_test.index, columns = num_variables)

X_train_num_scal.head()

,age,fnlwgt,education.num,capital.gain,capital.loss,hours.per.week
11356,-1.252217,-0.007188,-0.048105,-0.147593,-0.217634,-0.077299
11749,0.798626,-0.325029,-0.048105,-0.147593,-0.217634,2.013794
10504,-0.872431,0.294914,2.305328,-0.147593,-0.217634,-0.077299
7927,1.026497,-0.574963,-0.440343,-0.147593,-0.217634,-0.077299
18297,1.406283,0.521472,-0.048105,-0.147593,-0.217634,0.759138


Encoding

In [218]:
cat_variables = [
    "workclass",
    "education",
    "marital.status",
    "occupation",
    "relationship",
    "race",
    "sex",
    "native.country"
]

In [219]:
# instancio el encoder
onehot_encoder = OneHotEncoder(sparse_output=False,handle_unknown="ignore")

# entreno el encoder con los datos de entrenamiento
onehot_encoder.fit(X_train[cat_variables])

# aplico el encoder en amhos
X_train_cat_ohe = onehot_encoder.transform(X_train[cat_variables])
X_train_cat_ohe = pd.DataFrame(X_train_cat_ohe, index = X_train.index, columns=onehot_encoder.get_feature_names_out(cat_variables))

X_test_cat_ohe = onehot_encoder.transform(X_test[cat_variables])
X_test_cat_ohe = pd.DataFrame(X_test_cat_ohe, index = X_test.index, columns=onehot_encoder.get_feature_names_out(cat_variables))

X_train_cat_ohe.head()

,workclass_Federal-gov,workclass_Local-gov,workclass_Private,workclass_Self-emp-inc,workclass_Self-emp-not-inc,workclass_State-gov,workclass_Without-pay,education_10th,education_11th,education_12th,...,native.country_Portugal,native.country_Puerto-Rico,native.country_Scotland,native.country_South,native.country_Taiwan,native.country_Thailand,native.country_Trinadad&Tobago,native.country_United-States,native.country_Vietnam,native.country_Yugoslavia
11356,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
11749,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
10504,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
7927,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
18297,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0


In [220]:
# unificamos el dataset preprocesado hasta el momento
X_train_final = pd.concat([X_train_num_scal, X_train_cat_ohe], axis=1)
X_test_final = pd.concat([X_test_num_scal, X_test_cat_ohe], axis=1)

X_train_final.head()

,age,fnlwgt,education.num,capital.gain,capital.loss,hours.per.week,workclass_Federal-gov,workclass_Local-gov,workclass_Private,workclass_Self-emp-inc,...,native.country_Portugal,native.country_Puerto-Rico,native.country_Scotland,native.country_South,native.country_Taiwan,native.country_Thailand,native.country_Trinadad&Tobago,native.country_United-States,native.country_Vietnam,native.country_Yugoslavia
11356,-1.252217,-0.007188,-0.048105,-0.147593,-0.217634,-0.077299,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
11749,0.798626,-0.325029,-0.048105,-0.147593,-0.217634,2.013794,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
10504,-0.872431,0.294914,2.305328,-0.147593,-0.217634,-0.077299,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
7927,1.026497,-0.574963,-0.440343,-0.147593,-0.217634,-0.077299,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
18297,1.406283,0.521472,-0.048105,-0.147593,-0.217634,0.759138,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0


Conclusiones: ya se transformaron las variables categóricas y numéricas,

In [221]:
X_train_final.to_csv("../data/processed/X_train_final.csv", index=False)
X_test_final.to_csv("../data/processed/X_test_final.csv", index=False)

y_train.to_csv("../data/processed/y_train.csv", index=False)
y_test.to_csv("../data/processed/y_test.csv", index=False)

Modelo supervisado


In [222]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    random_state=42,
    max_iter=1000
)

model.fit(
    X_train_final,
    y_train
)

,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",42
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lb

In [223]:
y_pred = model.predict(
    X_test_final
)

In [224]:
accuracy = accuracy_score(
    y_test,
    y_pred)

print(accuracy)

0.8541804910418049


In [225]:
cm = confusion_matrix(
    y_test,
    y_pred)
cm

array([[4212,  315],
       [ 564,  937]])

In [226]:
print(classification_report(
    y_test,
    y_pred
))

              precision    recall  f1-score   support

           0       0.88      0.93      0.91      4527
           1       0.75      0.62      0.68      1501

    accuracy                           0.85      6028
   macro avg       0.82      0.78      0.79      6028
weighted avg       0.85      0.85      0.85      6028



Conclusiones:
Se entrenó un modelo de regresión logística para predecir si una persona supera los 50K$ anuales.
El modelo será utilizado como base para interpretar el perfil de los usuarios y construir recomendaciones.

El accuracy resultó 0.85 se considera que el rendimiento del modelo es bueno.

Sistemas de recomendación

1. Filtrado basado en contenido

Se utilizará filtrrado basado en contenido, el sistema comparará el perfil de los usuario scon perfiles similares entre sí.

- Se quieren recomendar trayectorias educativas y ocupacionales relacikonadas a ingresos superiores a 50k anuales.
- El usuario sería la persona con caraccterísticas laborales y socioeconómicas.
- La edad, educación, ocupación, estado civil, sexo , pais y horas trabajadas por seman definen el perfil

In [ ]:
#para usuarios con ingresos altos
X_train_altos = X_train_final[y_train == 1]

In [228]:
X_train_altos.shape

(6005, 103)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
def recomendar_por_contenido(usuario, n=5):
    
    #similitud entre el usuario elegido y los perfiles con ingresos altos
    sim_scores = cosine_similarity(usuario, X_train_altos).flatten()
    
    #ordenar de mayor a menor similitud
    sim_indices = sim_scores.argsort()[::-1][:n]
    
    #índices reales de los perfiles similares
    indices_reales = X_train_altos.iloc[sim_indices].index
    
    #perfiles similares del dataframe original
    perfiles_similares = df.loc[indices_reales]
    
    return perfiles_similares

probar con un usuario real

In [245]:
usuario = X_test_final.iloc[[0]]
indice_usuario = X_test_final.iloc[[0]].index[0]

df.loc[[indice_usuario]]

,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income
19117,27,Private,188171,10th,6,Never-married,Adm-clerical,Own-child,White,Male,0,0,60,United-States,0


Se pide usuarios similares al ususario de prueba.

In [ ]:
perfiles_similares = recomendar_por_contenido(usuario, n=5)

perfiles_similares[
    ["age", "education", "occupation", "hours.per.week", "income"]
]

,age,education,occupation,hours.per.week,income
10959,19,7th-8th,Other-service,60,1
11732,32,10th,Transport-moving,80,1
16650,22,11th,Machine-op-inspct,70,1
6986,39,10th,Craft-repair,60,1
5606,33,HS-grad,Prof-specialty,60,1


Conclusiones:
Se visualiza que el sistema ha dado perfiles similares deacuerdo al usuario de prueba escogido. 


De los perfiles similares, se detallan abajo las características más frecuentes de estos.


In [248]:
educacion_recomendada = perfiles_similares["education"].mode()[0]
ocupacion_recomendada = perfiles_similares["occupation"].mode()[0]
horas_recomendadas = round(perfiles_similares["hours.per.week"].mean(), 2)

print("educación más frecuente:", educacion_recomendada)
print("ocupación más frecuente:", ocupacion_recomendada)
print("horas promedio frecuentes:", horas_recomendadas)

educación más frecuente: 10th
ocupación más frecuente: Craft-repair
horas promedio frecuentes: 66.0


Conclusiones:
Se desarrolló un sistema de recomendación de tipo filtrado basado en contenido. el sistema idenntifica perfiles similares con ingresos superiores a 50k y recomienda opciones educatias y ocupacioales relacionadas a dichos perfiles.
